In [1]:
import pandas as pd
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document
import pickle

/home/kxelina/RAG_project/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv('Superstore.csv', encoding='cp1252')
df.drop(columns=['Row ID', 'Order ID', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'Postal Code', 'Product ID'], inplace=True)
df['Order Date'] = pd.to_datetime(df['Order Date'])

In [3]:
global_summaries = []
total_sales = df['Sales'].sum()
total_profit = df['Profit'].sum()
total_orders = len(df)
profit_margin = (total_profit / total_sales * 100) if total_sales > 0 else 0
global_summaries.append(f"GLOBAL SUMMARY: Total Sales ${total_sales:,.2f}, Total Profit ${total_profit:,.2f}, Total Orders {total_orders:,}, Overall Profit Margin {profit_margin:.2f}%")

# Top months by sales 
df['Month'] = df['Order Date'].dt.month
df['MonthName'] = df['Order Date'].dt.strftime('%B')
top_months = df.groupby(['Month', 'MonthName'])['Sales'].sum().reset_index().sort_values('Sales', ascending=False).head(3)
monthly_str = "TOP SALES MONTHS: " + ", ".join([f"{row['MonthName']} (${row['Sales']:,.2f})" for _, row in top_months.iterrows()])
global_summaries.append(monthly_str)

# Yearly totals
yearly_sales = df.groupby(df['Order Date'].dt.year).agg({
    'Sales': 'sum',
    'Profit': 'sum'
}).reset_index()
yearly_sales.columns = ['Year', 'Sales', 'Profit']

for _, row in yearly_sales.iterrows():
    year = int(row['Year'])
    sales = row['Sales']
    profit = row['Profit']
    margin = (profit / sales * 100) if sales > 0 else 0
    global_summaries.append(
        f"Year {year}: Sales ${sales:,.2f}, Profit ${profit:,.2f}, Margin {margin:.2f}%"
    )

# Category totals
for category in sorted(df['Category'].unique()):
    cat_df = df[df['Category'] == category]
    cat_sales = cat_df['Sales'].sum()
    cat_profit = cat_df['Profit'].sum()
    cat_margin = (cat_profit / cat_sales * 100) if cat_sales > 0 else 0
    global_summaries.append(
        f"{category}: Sales ${cat_sales:,.2f}, Profit ${cat_profit:,.2f}, Margin {cat_margin:.2f}%"
    )

# Region totals
for region in ['West', 'East', 'Central', 'South']:
    region_df = df[df['Region'] == region]
    if len(region_df) > 0:
        region_sales = region_df['Sales'].sum()
        region_profit = region_df['Profit'].sum()
        region_margin = (region_profit / region_sales * 100) if region_sales > 0 else 0
        global_summaries.append(
            f"{region}: Sales ${region_sales:,.2f}, Profit ${region_profit:,.2f}, Margin {region_margin:.2f}%"
        )

# Sub-category totals 
for sub_cat in sorted(df['Sub-Category'].unique()):
    sub_df = df[df['Sub-Category'] == sub_cat]
    sub_sales = sub_df['Sales'].sum()
    sub_profit = sub_df['Profit'].sum()
    sub_margin = (sub_profit / sub_sales * 100) if sub_sales > 0 else 0
    items_count = len(sub_df)
    discounted_count = len(sub_df[sub_df['Discount'] > 0])
    global_summaries.append(
        f"Sub-Category {sub_cat}: ${sub_sales:,.2f} sales, {sub_margin:.2f}% margin, {items_count} items, {discounted_count} discounted"
    )

# Top states by sales 
state_sales = df.groupby('State').agg({'Sales': 'sum', 'Profit': 'sum'}).reset_index()
state_sales = state_sales.sort_values('Sales', ascending=False).head(10)
for _, row in state_sales.iterrows():
    state = row['State']
    state_s = row['Sales']
    state_p = row['Profit']
    state_m = (state_p / state_s * 100) if state_s > 0 else 0
    global_summaries.append(
        f"State {state}: ${state_s:,.2f} sales, ${state_p:,.2f} profit, {state_m:.2f}% margin"
    )

# Top cities by sales 
city_sales = df.groupby('City').agg({'Sales': 'sum', 'Profit': 'sum'}).reset_index()
city_sales = city_sales.sort_values('Sales', ascending=False).head(15)
for _, row in city_sales.iterrows():
    city = row['City']
    city_s = row['Sales']
    city_p = row['Profit']
    city_m = (city_p / city_s * 100) if city_s > 0 else 0
    global_summaries.append(
        f"City {city}: ${city_s:,.2f} sales, ${city_p:,.2f} profit, {city_m:.2f}% margin"
    )

print(f"✓ Created {len(global_summaries)} global summary documents")

✓ Created 55 global summary documents


In [4]:
grouped_summaries = []

# Year-Category combinations
for year in sorted(df['Order Date'].dt.year.unique()):
    for category in sorted(df['Category'].unique()):
        subset = df[(df['Order Date'].dt.year == year) & (df['Category'] == category)]
        if len(subset) > 0:
            sales = subset['Sales'].sum()
            profit = subset['Profit'].sum()
            margin = (profit / sales * 100) if sales > 0 else 0
            grouped_summaries.append(
                f"{year} {category}: ${sales:,.2f} sales, ${profit:,.2f} profit, {margin:.2f}% margin"
            )

# Year-SubCategory combinations 
for year in sorted(df['Order Date'].dt.year.unique()):
    for sub_cat in sorted(df['Sub-Category'].unique()):
        subset = df[(df['Order Date'].dt.year == year) & (df['Sub-Category'] == sub_cat)]
        if len(subset) > 0:
            sales = subset['Sales'].sum()
            profit = subset['Profit'].sum()
            margin = (profit / sales * 100) if sales > 0 else 0
            grouped_summaries.append(
                f"{year} {sub_cat}: ${sales:,.2f} sales, {margin:.2f}% margin"
            )

# Year-Region combinations
for year in sorted(df['Order Date'].dt.year.unique()):
    for region in ['West', 'East', 'Central', 'South']:
        subset = df[(df['Order Date'].dt.year == year) & (df['Region'] == region)]
        if len(subset) > 0:
            sales = subset['Sales'].sum()
            profit = subset['Profit'].sum()
            grouped_summaries.append(
                f"{year} {region}: ${sales:,.2f} sales, ${profit:,.2f} profit"
            )

# Year-State combinations 
for year in sorted(df['Order Date'].dt.year.unique()):
    state_data = df[df['Order Date'].dt.year == year].groupby('State').agg({'Sales': 'sum', 'Profit': 'sum'}).reset_index()
    for _, row in state_data.iterrows():
        state_s = row['Sales']
        state_p = row['Profit']
        grouped_summaries.append(
            f"{year} {row['State']}: ${state_s:,.2f} sales, ${state_p:,.2f} profit"
        )

# Monthly summaries with category detail 
df['YearMonth'] = df['Order Date'].dt.to_period('M')
for period in sorted(df['YearMonth'].unique()):
    subset = df[df['YearMonth'] == period]
    sales = subset['Sales'].sum()
    profit = subset['Profit'].sum()
    grouped_summaries.append(
        f"{period.strftime('%B %Y')}: ${sales:,.2f} sales, ${profit:,.2f} profit"
    )

# Discount analysis 
discount_analysis = df[df['Discount'] > 0].groupby('Sub-Category').agg({
    'Discount': 'count',
    'Sales': 'sum'
}).reset_index()
discount_analysis.columns = ['Sub-Category', 'Items_Discounted', 'Total_Sales']
for _, row in discount_analysis.iterrows():
    grouped_summaries.append(
        f"Discount: {row['Sub-Category']} - {row['Items_Discounted']} items at discount, ${row['Total_Sales']:,.2f} in sales"
    )

# High-discount products (>20% off)
high_discount = df[df['Discount'] >= 0.2].groupby('Sub-Category').size().reset_index(name='count')
total_high_discount = high_discount['count'].sum()
grouped_summaries.append(f"Products with >20% discount: {total_high_discount} items across all categories")



In [5]:
raw_transactions = []

for index, row in df.iterrows():
    year = row['Order Date'].year
    category = row['Category']
    sub_category = row['Sub-Category']
    city = row['City']
    state = row['State']
    region = row['Region']
    sales = f"{row['Sales']:.2f}"
    profit = f"{row['Profit']:.2f}"
    discount = f"{row['Discount']*100:.0f}%" if row['Discount'] > 0 else "0%"
    qty = int(row['Quantity'])
    
    
    compressed = f"{year}|{category}|{sub_category}|{city}|{state}|{region}|Qty:{qty}|Sales:${sales}|Profit:${profit}|Disc:{discount}"
    raw_transactions.append(compressed)

print(f"✓ Created {len(raw_transactions)} compressed transactions")


✓ Created 9994 compressed transactions


In [6]:
def smart_chunk_summaries(summaries, chunk_size=400, chunk_overlap=50, layer_name="global"):
    """
    Intelligently chunk summary documents with overlap.
    - Groups related summaries together before chunking
    - Preserves semantic meaning through overlap
    - Adds rich metadata for filtering
    """
    docs = []
    chunk_id = 0
    
    # Generic approach: join summaries smartly, then chunk
    text = "\n".join(summaries)
    
    splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )
    
    chunks = splitter.split_text(text)
    
    for chunk in chunks:
        # Extract fact type from chunk content (year, category, region, sub-cat, state, city)
        fact_type = "metric"
        if "Year " in chunk and "$" in chunk:
            fact_type = "yearly"
        elif "Sub-Category" in chunk:
            fact_type = "subcategory"
        elif "State " in chunk:
            fact_type = "state"
        elif "City " in chunk:
            fact_type = "city"
        elif "Region" in chunk or any(r in chunk for r in ['West', 'East', 'Central', 'South']):
            fact_type = "region"
        elif "Category" in chunk and layer_name == "global":
            fact_type = "category"
        
        doc = Document(
            page_content=chunk,
            metadata={
                "chunk_id": chunk_id,
                "layer": layer_name,
                "chunk_size": len(chunk),
                "fact_type": fact_type
            }
        )
        docs.append(doc)
        chunk_id += 1
    
    return docs

# CHUNK ALL LAYERS for better retrieval
print("Chunking with intelligent grouping and overlap...\n")

# Global summaries: chunk by 400 chars with 50 char overlap
global_docs = smart_chunk_summaries(global_summaries, chunk_size=400, chunk_overlap=50, layer_name="global")

# Grouped summaries: chunk by 350 chars with 50 char overlap  
grouped_docs = smart_chunk_summaries(grouped_summaries, chunk_size=350, chunk_overlap=50, layer_name="grouped")

# Raw transactions: chunk by 600 chars with 30 char overlap (transactions naturally fit this size)
text = "\n".join(raw_transactions)
splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=600,
    chunk_overlap=30,
    length_function=len
)
chunks = splitter.split_text(text)
raw_docs = []
for i, chunk in enumerate(chunks):
    doc = Document(
        page_content=chunk,
        metadata={"chunk_id": i, "layer": "raw", "chunk_size": len(chunk), "fact_type": "transaction"}
    )
    raw_docs.append(doc)



Chunking with intelligent grouping and overlap...



In [7]:
# Save with layer information
improved_data = {
    "global": {"documents": global_docs},
    "grouped": {"documents": grouped_docs},
    "raw": {"documents": raw_docs}
}

with open("preprocessing_improved.pkl", "wb") as f:
    pickle.dump(improved_data, f)
print(f"Saved: {len(global_docs)} global + {len(grouped_docs)} grouped + {len(raw_docs)} raw = {len(global_docs) + len(grouped_docs) + len(raw_docs)} total")

Saved: 12 global + 58 grouped + 1668 raw = 1738 total
